# Digital Cash Management Forecast: set up the project

This notebook sets up the Digital Cash Management Forecast in this Dataiku project (it also gives the project
that name), the way it runs in production. Everything the project writes is a CSV file (compressed for the big ones).
It reads 2 warehouse tables, read-only, and creates or writes nothing in the warehouse: the clients
(`EDW.DIM_CUSTOMER`) and the daily postings (`EDW.FCT_STATEMENT_ENTRY`), which the staging recipe sums per client,
day and category in Dataiku. The first run reads each client's whole history; after that, each run reads only the
new days (through the database connection, a `SELECT` with a date filter) and adds them to `daily_flows`. In the sandbox, they are the datasets the notebook `warehouse_simulator.ipynb` wrote in
this project; in production, the warehouse's tables through the read-only database connection
`WAREHOUSE_SQL_CONNECTION` (or CSV files in the landing folder). Names, codes and the categories (a mock list until
the bank's transaction types are known): `python/warehouse.py` in the project library.

It forecasts, per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 3 months, as P10, EV and P90. It builds 5 flow zones, each with an owner:

- **Raw data** and **Staging** (shared): the 2 warehouse tables, read-only, then `clients` (the business clients)
  and `daily_flows` (money in and out per client, day and category): what every model gets. `staging_log` records
  each read.
- **Training** (data scientist): each training run is one model with its own **model ID**, trained with the
  **forecaster** the project variable `forecaster` names, tested on the last 6 months, tracked in Dataiku's
  **Experiment Tracking** and deployed as a new **version** of the saved model `cash_forecast`. A champion /
  challenger test activates the new version if it passes the **promotion rules** (project variables `promotion_*`).
- **Inference** (inference owner): `final_forecast`, the production model's forecast for every active client, made
  a chunk of clients at a time so it scales to the bank's ~420k clients (every row says which model and which load
  produced it), the forecast archive, and the inference log (one row per run, with timings and memory).
- **Lab** (data scientist): `try_forecaster` tries the forecaster the project variable `lab_forecaster` names
  against the production model and adds the result to `lab_results`, without touching production. The notebook
  `forecast_lab.ipynb` does the same interactively.

The zones only meet through three interfaces: **1. the data** (`daily_flows` and `clients`: staging → training, lab
and inference), **2. the forecaster** (`models.py`: lab → training), **3. the active version of the saved model**
(training → inference). Until the data scientist's models are ready, the forecaster is a placeholder,
`recent_average` (the average of the last 3 months, no machine learning). Two independent scenarios run it:
**Nightly batch** (inference) and **Retrain, test and promote** (training). Each starts by setting the cut-off month
(`set_cutoff`: the month before today, or the project variable `cutoff_override`); the staging recipe stops if the
postings do not reach it.

It then trains, tests and promotes a first model, and runs the first nightly batch.

**How to use it:** in the project, import this notebook and `warehouse_simulator.ipynb` (Notebooks > New notebook
> Import). In the sandbox, run the simulator first. Then check the settings below and run all cells (Cell > Run All):
a few minutes with the small sample data. It can be run again in the same project: what exists is kept (data,
saved model versions, zones, scenario history), and the recipes' code is updated.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast"
WAREHOUSE_SQL_CONNECTION = ""         # production: the read-only database connection to the warehouse (Oracle), to
WAREHOUSE_SQL_SCHEMA = "EDW"          # read DIM_CUSTOMER and FCT_STATEMENT_ENTRY of this schema as they are. Empty:
LANDING_CONNECTION = "EDW_LANDING"    # read them as CSV files from the landing folder (read-only connection). Neither
                                      # is used for tables the warehouse simulator has written here (sandbox: CSV).
CONNECTION = "filesystem_managed"     # where the project writes its own datasets (CSV files)
FOLDER_CONNECTION = "filesystem_folders"   # where the tracked models and the forecast archive are kept
CODE_ENV = "cashflow_forecast"        # Python code env for the recipes: an existing one (an admin creates it once)

## 1. Code environment

The recipes need scikit-learn, and MLflow for Dataiku's Experiment Tracking, in an existing code env (`CODE_ENV`
above). This notebook does not create it: an admin creates it once. This checks it is there with those packages,
and otherwise stops with the request to send the admin.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())
# scikit-learn: for the data scientist's models; mlflow: Experiment Tracking and the saved model. Dataiku tested MLflow models with
# mlflow 2.22.0 and 3.10.1 (DSS 14 and 15 docs, "MLflow models: limitations"); mlflow 3 needs Python 3.10 or later.
# Either works here; the request below asks for 2.22.0, which also runs on Python 3.9.
PACKAGES = ("scikit-learn", "mlflow==2.22.0")


def check_code_env(name, packages=PACKAGES):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.9 or later) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. {request}")
    print(f"Code env {name} has {', '.join(packages)}")


check_code_env(CODE_ENV)

## 2. Code

The project library gets these files, by owner:

- shared: `warehouse.py` (the warehouse tables and columns read, codes, categories, daily sums, cut-off month) and
  `cashflow.py` (dates, the data interface, days to months, the saved model name)
- training and lab: `models.py` (the forecaster interface, the placeholder forecaster, the test and its scores) and
  `tracking.py` (model IDs, Experiment Tracking, saved model versions, promotion rules, champion / challenger)
- inference: `forecast.py` (the production model, the final forecast table)
- `forecasters.py`: the data scientist's own forecasters. Written only when it is created: running this notebook
  again never overwrites it.

Each recipe below becomes a Python recipe in the flow.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

The forecast reads 2 tables of the bank's warehouse, read-only, and creates or writes nothing there:
    DIM_CUSTOMER          EDW.DIM_CUSTOMER: the clients. The project keeps the business clients (companies and sole
                          proprietors) and only the columns below: personal fields are never read.
    FCT_STATEMENT_ENTRY   EDW.FCT_STATEMENT_ENTRY: the daily postings. The staging recipe sums them per client, day
                          and category (money in, money out) in Dataiku and adds the new days to daily_flows. Through
                          the database connection, the only query is a read-only SELECT of the columns below with a
                          date filter on VAL_DT (transactions_query): the whole history the first time (or after the
                          scenario Full rebuild), then only the days after the last day already in daily_flows.
Through the database connection to the warehouse (Oracle, read-only), or as CSV files in a landing folder; the setup
notebook's settings choose. The sandbox's simulated tables are CSV (read whole, filtered by date in Dataiku). If a
name or a code differs at the bank, only this file changes.
"""
import numpy as np
import pandas as pd

# the tables the project reads. standard name: (table, where it is)
TABLES = {
    "clients": ("DIM_CUSTOMER", "EDW.DIM_CUSTOMER"),
    "transactions": ("FCT_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
}
STAGED = ["clients", "daily_flows", "staging_log"]       # what the staging recipe writes

# Categories: each posting gets one from its transaction type (TRNS_TYP); a type not listed goes to OTHER.
# MOCK LIST: these are the sample data's types. At the bank, replace it with the real TRNS_TYP codes (ask the
# warehouse team for the list of codes and what each means); nothing else changes.
CATEGORY_OF_TYPE = {
    "COLLECTION": "CUSTOMER_RECEIPTS", "CARD_SETTLEMENT": "CUSTOMER_RECEIPTS", "CHEQUE": "CUSTOMER_RECEIPTS",
    "SALARY": "SALARIES", "RENT": "RENT", "SUPPLIER": "SUPPLIERS", "FINANCING": "FINANCING", "GOV": "GOVERNMENT",
    "UTILITY": "UTILITIES", "CARD": "CARDS", "FEE": "FEES",
}
OTHER = "OTHER"

# the columns read. standard name: warehouse (EDW) name
COLUMNS = {
    "clients": {"client_no": "CSTMR_CIF", "customer_type": "CSTMR_TYP", "line_of_business": "LOB_DSC",
                "legal_entity": "LGL_ENTY_DSC", "sub_legal_entity": "PRTY_SUB_LGL_DSC", "id_type": "PRTY_ID_TYP_DSC",
                "segment": "RM_SGMNT_NM", "sector": "SCTR_DSC", "business_sector": "BUS_SCTR_EN_NM",
                "industry": "INDST_EN_NM", "sub_industry": "SUB_INDST_EN_NM", "city": "CITY", "status": "STS_FLG"},
    # one row per posting. AMT_LCY: SAR, negative for money out (T24). TRNS_TYP: the transaction type, which gives
    # the category. RVRSL_MRKR: set on a reversing posting, which carries the reversed posting's TRNS_RFR
    "transactions": {"entry_id": "STMT_ENTRY_ID", "client_no": "CSTMR_CIF", "account_no": "ACNT_NMBR",
                     "value_date": "VAL_DT", "direction": "CRDT_DBT_FLG", "amount": "AMT_LCY", "txn_type": "TRNS_TYP",
                     "txn_ref": "TRNS_RFR", "reversal": "RVRSL_MRKR"},
}
# codes in DIM_CUSTOMER (from the bank's data profile; to confirm on business clients)
ACTIVE_STATUS = ("ACTIVE", "A")               # STS_FLG of an active client (an inactive one shows INACTIVE)
RETAIL = ("RETAIL",)                          # words of LOB_DSC for retail (individual) clients: left out...
BUSINESS_ID = ("COMMERCIAL",)                 # ...unless their ID type (PRTY_ID_TYP_DSC) is a commercial registration
SOLE_PROPRIETOR = ("SOLE", "ESTABLISHMENT")   # words of LGL_ENTY_DSC or PRTY_SUB_LGL_DSC for a business owned by 1 person


def read_columns(table):
    """The warehouse columns the project reads from a table."""
    return list(COLUMNS[table].values())


def to_standard(df, table):
    """A warehouse table with the standard column names and the project's values: client numbers as text, direction
    IN / OUT; for clients, status ACTIVE / CLOSED and sole_proprietor true / false. A missing column stops here
    with its name."""
    missing = [c for c in COLUMNS[table].values() if c not in df]
    if missing:
        raise ValueError(f"{TABLES[table][0]} has no column {', '.join(missing)}: check the warehouse table")
    out = pd.DataFrame({standard: df[column] for standard, column in COLUMNS[table].items()})
    out["client_no"] = out["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "direction" in out:
        out["direction"] = out["direction"].astype(str).str.strip().str.upper().str[0].map({"C": "IN", "D": "OUT"})
    if table == "clients":
        active = out["status"].astype(str).str.strip().str.upper().isin(ACTIVE_STATUS)
        out["status"] = np.where(active, "ACTIVE", "CLOSED")
        legal = (out["legal_entity"].fillna("").astype(str) + " " + out["sub_legal_entity"].fillna("").astype(str))
        out["sole_proprietor"] = legal.str.upper().str.contains("|".join(SOLE_PROPRIETOR))
    return out


def business_clients(clients):
    """The business clients (standard names): every client whose line of business does not contain RETAIL, plus the retail
    ones registered with a commercial registration (sole proprietors). One row per client number."""
    lob = clients["line_of_business"].fillna("").astype(str).str.upper()
    id_type = clients["id_type"].fillna("").astype(str).str.upper()
    business = ~lob.str.contains("|".join(RETAIL)) | id_type.str.contains("|".join(BUSINESS_ID))
    return clients[business].drop_duplicates("client_no", keep="last").reset_index(drop=True)


def value_day(dates):
    """The day of each value date, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.normalize()


def transactions_query(table, since, end):
    """The read-only query staging runs through the database connection: the columns read (COLUMNS) of the
    postings with a value date from since (None: the whole history) to before end. table: the table's full quoted
    name. The value date comes back as text YYYY-MM-DD (TO_CHAR: Oracle and PostgreSQL), so no time zone can move
    it."""
    date = COLUMNS["transactions"]["value_date"]
    columns = [f"TO_CHAR(\"{date}\", 'YYYY-MM-DD') AS \"{date}\"" if c == date else f'"{c}"'
               for c in COLUMNS["transactions"].values()]
    where = [f"\"{date}\" < DATE '{end:%Y-%m-%d}'"]
    if since is not None:
        where.insert(0, f"\"{date}\" >= DATE '{since:%Y-%m-%d}'")
    return f"SELECT {', '.join(columns)} FROM {table} WHERE {' AND '.join(where)}"


def category(txn_type):
    """The category of each posting, from its transaction type (CATEGORY_OF_TYPE; OTHER if not listed)."""
    return txn_type.fillna("").astype(str).str.strip().str.upper().map(CATEGORY_OF_TYPE).fillna(OTHER)


def daily_flows(entries, client_numbers, cutoff):
    """The daily totals of postings (standard names): one row per client, day and category with money_in, money_out
    (both positive, SAR) and the number of postings. Business clients only, every day up to the end of the cut-off
    month, without reversed postings (nor what they reverse) and without transfers between the client's own accounts
    (a debit and a credit with the same TRNS_RFR on the same client, netting to zero). The staging recipe calls it on
    postings grouped so that all the postings of a TRNS_RFR are together."""
    e = entries[entries["client_no"].isin(set(client_numbers))].copy()
    e["day"] = value_day(e["value_date"])
    e = e[e["day"] < cutoff + pd.DateOffset(months=1)]
    marked = e["reversal"].notna() & ~e["reversal"].astype(str).str.strip().isin(["", "nan", "None"])
    e = e[~e["txn_ref"].isin(set(e.loc[marked, "txn_ref"].dropna()))]
    legs = e[e["txn_ref"].notna()].groupby(["txn_ref", "client_no"]).agg(sides=("direction", "nunique"),
                                                                         net=("amount", "sum"))
    own = legs[(legs["sides"] == 2) & (legs["net"].abs() < 0.005)].index
    e = e[~pd.MultiIndex.from_frame(e[["txn_ref", "client_no"]]).isin(own)]
    e = e.assign(category=category(e["txn_type"]), amount=e["amount"].abs(),
                 money_in=lambda d: d["amount"].where(d["direction"] == "IN", 0.0),
                 money_out=lambda d: d["amount"].where(d["direction"] == "OUT", 0.0))
    return (e.groupby(["client_no", "day", "category"], as_index=False)
            .agg(money_in=("money_in", "sum"), money_out=("money_out", "sum"), postings=("entry_id", "count")))


def set_cutoff():
    """The forecast's cut-off: the last complete month, by default the month before today. The project variable
    cutoff_override (YYYY-MM) fixes it instead: the sandbox's simulator sets it, and it reruns a past month. Sets the
    project variables cutoff_month, as_of_date and load_id (the data's as-of date), which the recipes read. The
    staging recipe stops if the postings do not reach the cut-off month (a late warehouse load).
    Run by the first step of each scenario: from warehouse import set_cutoff; set_cutoff()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    override = variables["standard"].get("cutoff_override")
    cutoff = pd.Period(override, "M") if override else pd.Timestamp.now().to_period("M") - 1
    as_of = cutoff.to_timestamp(how="end").normalize()
    variables["standard"].update(cutoff_month=str(cutoff), as_of_date=f"{as_of:%Y-%m-%d}",
                                 load_id=f"EDW_{as_of:%Y%m%d}")
    project.set_variables(variables)
    print(f"Cut-off month {cutoff} (data as of {as_of:%Y-%m-%d})" + (" (cutoff_override)" if override else ""))
    return str(cutoff)


def clear_daily_flows():
    """First step of the scenario Full rebuild (after the cut-off): empties daily_flows and staging_log, so that the
    staging recipe reads the whole history again. Run as: from warehouse import clear_daily_flows; clear_daily_flows()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    for name in ("daily_flows", "staging_log"):
        project.get_dataset(name).clear()
    print("daily_flows and staging_log emptied: the staging recipe will read the whole history")


for _table, _columns in COLUMNS.items():          # warehouse names unique within a table
    assert len(set(_columns.values())) == len(_columns), _table
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared by every zone (project library: python/cashflow.py): dates, the business rule on amounts, and the
interfaces between the zones:
    1. the data (staging -> training, lab and inference): the datasets daily_flows (one row per client, day and
       category: money_in, money_out, postings, over each client's whole history) and clients (one row per business
       client). Every model gets these and nothing else; monthly_matrix and monthly_series below turn the days into
       months for the models that work by month.
    2. the forecaster (lab -> training): models.py.
    3. the saved model (training -> inference): SAVED_MODEL; inference uses its active version.
The forecast: per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 3 months, as P10, EV and P90 in SAR."""
import numpy as np
import pandas as pd

HORIZONS = (1, 2, 3)        # months ahead
TOTAL = "TOTAL"             # the category of a client's all-categories net cash flow
SAVED_MODEL = "cash_forecast"
MIN_HISTORY_MONTHS = 12     # a client needs this many months before the cut-off to get a forecast (else status
                            # NOT_ENOUGH_HISTORY in the final forecast)
FORECAST_COLUMNS = ["client_no", "category", "months_ahead", "p10", "ev", "p90"]   # what a forecaster returns


def _plain(df):
    """Dates as plain (time zone free) timestamps, client numbers as text."""
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    if "client_no" in df:
        df["client_no"] = df["client_no"].astype(str)
    return df


def read_dataset(name, columns=None):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps and client numbers as text."""
    import dataiku
    return _plain(dataiku.Dataset(name).get_dataframe(columns=columns))


def iter_dataset(name, rows=1_000_000, columns=None, **kwargs):
    """A dataset read in chunks of rows (DataFrames, like read_dataset), so a big one never sits in memory whole."""
    import dataiku
    for chunk in dataiku.Dataset(name, **kwargs).iter_dataframes(chunksize=rows, columns=columns):
        yield _plain(chunk)


class Spread:
    """Rows spread over n files on local disk by the hash of a column, then read back one file at a time: all the
    rows with the same value of that column end up in the same file, and memory holds one file, not the whole table.
    Used by staging (postings by reference, daily totals by client) and inference (daily flows by client)."""

    def __init__(self, folder, n):
        import os
        os.makedirs(folder, exist_ok=True)
        self.n = max(1, int(n))
        self.paths = [os.path.join(folder, f"part_{i}.pkl") for i in range(self.n)]

    def add(self, df, key):
        """key: a column name, or a Series (same index as df) to spread by."""
        import pickle
        key = df[key] if isinstance(key, str) else key
        part = pd.util.hash_pandas_object(key.astype(str), index=False).to_numpy() % self.n
        for i, rows in df.groupby(part):
            with open(self.paths[i], "ab") as f:
                pickle.dump(rows, f, protocol=4)

    def parts(self):
        """Each file's rows (a DataFrame); files that got no rows are skipped."""
        import os
        import pickle
        for path in self.paths:
            if not os.path.exists(path):
                continue
            frames = []
            with open(path, "rb") as f:
                while True:
                    try:
                        frames.append(pickle.load(f))
                    except EOFError:
                        break
            yield pd.concat(frames, ignore_index=True)


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def add_months(month, n):
    return month + pd.offsets.DateOffset(months=n)


def cutoff_month():
    """The forecast's cut-off: the last complete month (project variable
    cutoff_month, set by the first step of each scenario: warehouse.set_cutoff)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_month"] + "-01")


def until(daily, last_month):
    """The daily flows known at the end of last_month (the days up to its last day)."""
    return daily[pd.to_datetime(daily["day"]) < add_months(last_month, 1)]


def finish(ev, p10, p90, index=None):
    """P10 <= EV <= P90 (SAR). The net cash flow can be below zero."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    return pd.DataFrame({"p10": np.minimum(p10, ev), "ev": ev, "p90": np.maximum(p90, ev)}, index=index).round(2)


def monthly_matrix(daily, last_month):
    """The net cash flow (money in minus money out) per month: one row per client and category (and per client in
    TOTAL), one column per month up to last_month. 0 for a month with no posting after the series' first month,
    empty before it. Only the days up to the end of last_month are used."""
    d = until(daily, last_month)
    d = d.assign(month=month_start(d["day"]).values, net=(d["money_in"] - d["money_out"]).values)
    by_category = d.groupby(["client_no", "category", "month"])["net"].sum()
    total = d.groupby(["client_no", "month"])["net"].sum().to_frame().assign(category=TOTAL)
    total = total.set_index("category", append=True).reorder_levels(["client_no", "category", "month"])["net"]
    sums = pd.concat([by_category, total])
    if sums.empty:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=["client_no", "category"]))
    wide = sums.unstack("month")
    months = pd.date_range(wide.columns.min(), last_month, freq="MS")
    wide = wide.reindex(columns=months)
    started = wide.notna().cumsum(axis=1) > 0
    return wide.fillna(0.0).where(started).sort_index()


def monthly_series(daily, last_month):
    """monthly_matrix as rows: client_no, category, month, net (no row before a series' first month)."""
    long = monthly_matrix(daily, last_month).reset_index().melt(id_vars=["client_no", "category"], var_name="month",
                                                                value_name="net")
    long["month"] = pd.to_datetime(long["month"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "month"]).reset_index(drop=True)


def history_months(daily, last_month):
    """Per client: the months from its first posting to last_month (both counted)."""
    first = month_start(until(daily, last_month).groupby("client_no")["day"].min())
    months = (last_month.year - first.dt.year) * 12 + (last_month.month - first.dt.month) + 1
    return months.rename("history_months")
'''

In [ ]:
MODELS_CODE = r'''
"""Models (project library: python/models.py): interface 2, the forecaster, the placeholder forecaster and the test
every model goes through. Training (train_and_test, champion_challenger) and the lab use it; inference only calls the
production model's predict.

A forecaster is a class with:
    name            its name. The project variable forecaster says which one the scenario Retrain, test and promote
                    trains (default: recent_average).
    params          its settings (a dict), recorded in Experiment Tracking
    fit(daily, clients, cutoff)
                    learns from the data known at the end of the cut-off month (interface 1, cashflow.py):
                    daily: daily_flows (client_no, day, category, money_in, money_out, postings), every day up to
                    the end of the cut-off month; clients: the clients dataset; cutoff: the cut-off month (a
                    Timestamp, first day of the month). cashflow.monthly_matrix turns the days into months.
    predict(daily, clients, cutoff)
                    the same inputs, for the clients to forecast. Returns a DataFrame with one row per client,
                    category (each category the client has in daily, and TOTAL: all its categories together) and
                    months ahead (1, 2, 3): the columns client_no, category, months_ahead, p10, ev, p90 (the net cash
                    flow of that month, SAR). forecast() checks it and applies P10 <= EV <= P90.
The data scientist's own forecasters go in forecasters.py: the setup notebook creates that file once and never
overwrites it. lab_try and lab_compare try forecasters against the production model without touching production.
"""
import numpy as np
import pandas as pd

from cashflow import FORECAST_COLUMNS, HORIZONS, add_months, cutoff_month, finish, monthly_matrix, monthly_series, until

HOLDOUT_MONTHS = 6          # the last months kept aside to test the models
OUTPUT = ["p10", "ev", "p90"]


class RecentAverage:
    """The placeholder forecaster, until the data scientist's models replace it: no machine learning. Per client and
    category, EV = the average net cash flow of the last 3 months (the same for the 3 months ahead). P10 and P90 =
    EV -/+ width x the series' standard deviation over the last 12 months, where fit sets width so that, over the
    12 months before the cut-off, 80% of the actual months fell inside."""
    name = "recent_average"
    params = {"months": 3, "spread_months": 12, "coverage": 0.8}

    def fit(self, daily, clients, cutoff):
        wide = monthly_matrix(daily, cutoff)
        n, s = self.params["months"], self.params["spread_months"]
        z = []
        for k in range(1, min(12, wide.shape[1] - 1) + 1):         # each of the last 12 months, from the months before
            past, actual = wide.iloc[:, :-k], wide.iloc[:, -k]
            ev, sd = past.iloc[:, -n:].mean(axis=1), past.iloc[:, -s:].std(axis=1)
            ok = actual.notna() & ev.notna() & (sd > 0)
            z.append(((actual - ev).abs() / sd)[ok])
        z = pd.concat(z) if z else pd.Series(dtype=float)
        self.width = float(z.quantile(self.params["coverage"])) if len(z) else 1.28
        return self

    def predict(self, daily, clients, cutoff):
        wide = monthly_matrix(daily, cutoff)
        if wide.empty:
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        ev = wide.iloc[:, -self.params["months"]:].mean(axis=1).fillna(0.0)
        sd = wide.iloc[:, -self.params["spread_months"]:].std(axis=1).fillna(ev.abs())
        one = pd.DataFrame({"p10": ev - self.width * sd, "ev": ev, "p90": ev + self.width * sd}).reset_index()
        return pd.concat([one.assign(months_ahead=h) for h in HORIZONS], ignore_index=True)[FORECAST_COLUMNS]


def known_forecasters():
    """The forecasters Retrain can train, by name: the placeholder and those listed in FORECASTERS in forecasters.py.
    Also returns the problem reading forecasters.py, if any (the placeholder still works)."""
    found, problem = {RecentAverage.name: RecentAverage}, None
    try:
        import forecasters
        for forecaster in forecasters.FORECASTERS:
            found[forecaster.name] = forecaster
    except Exception as e:
        problem = f"forecasters.py could not be read ({type(e).__name__}: {e})"
    return found, problem


def new_forecaster(name=None):
    """A new, untrained forecaster: the one named by the project variable forecaster (default recent_average)."""
    import dataiku
    name = name or dataiku.get_custom_variables().get("forecaster") or RecentAverage.name
    found, problem = known_forecasters()
    if name not in found:
        raise ValueError(f"No forecaster named {name!r} (project variable forecaster). Known: {', '.join(found)}. "
                         + (problem if problem else "Add yours to FORECASTERS in forecasters.py."))
    forecaster = found[name]()
    missing = [a for a in ("name", "fit", "predict") if not hasattr(forecaster, a)]
    if missing:
        raise ValueError(f"Forecaster {name} has no {', '.join(missing)} (see models.py)")
    return forecaster


def forecast(forecaster, daily, clients, cutoff):
    """P10, EV and P90 of a trained forecaster (SAR), checked, with P10 <= EV <= P90: one row per client, category and
    months ahead. Everything that predicts goes through here: the test, champion / challenger, the lab and the
    nightly forecast."""
    out = forecaster.predict(daily, clients, cutoff)
    name = getattr(forecaster, "name", type(forecaster).__name__)
    if not isinstance(out, pd.DataFrame) or not set(FORECAST_COLUMNS) <= set(out.columns):
        raise ValueError(f"Forecaster {name}: predict must return a DataFrame with the columns "
                         f"{', '.join(FORECAST_COLUMNS)}")
    empty = [c for c in OUTPUT if out[c].isna().any()]
    if empty:
        raise ValueError(f"Forecaster {name}: predict returned empty values in {', '.join(empty)}")
    out = out[FORECAST_COLUMNS].reset_index(drop=True)
    out["client_no"] = out["client_no"].astype(str)
    out["months_ahead"] = out["months_ahead"].astype(int)
    out[OUTPUT] = finish(out["ev"], out["p10"], out["p90"], out.index)
    return out.sort_values(["client_no", "category", "months_ahead"]).reset_index(drop=True)


def active(daily, clients):
    """The active clients and their daily flows: the clients the test and the forecast are made for."""
    clients = clients[clients["status"] == "ACTIVE"]
    return daily[daily["client_no"].isin(set(clients["client_no"]))], clients


def test_forecasts(fitted, daily, clients, cutoff):
    """The test, the same for every model: a forecaster trained without the last 6 months forecasts each of those
    months from the data known the month before (cut-offs: the 6 months before the cut-off), for the active clients.
    Returns one row per client, category, test cut-off and months ahead whose predicted month is known: target (the
    actual net cash flow), p10, ev, p90 and baseline (the simple guess: the same as the month before the forecast)."""
    daily, clients = active(daily, clients)
    known = monthly_series(daily, cutoff)
    actual = known.rename(columns={"month": "target_month", "net": "target"})
    last = known.rename(columns={"month": "cutoff_month", "net": "baseline"})
    parts = []
    for k in range(HOLDOUT_MONTHS):
        t = add_months(cutoff, k - HOLDOUT_MONTHS)
        f = forecast(fitted, until(daily, t), clients, t).assign(cutoff_month=t)
        f["target_month"] = f["months_ahead"].map({h: add_months(t, h) for h in HORIZONS})
        parts.append(f[f["target_month"] <= cutoff])
    keys = ["client_no", "category"]
    tested = (pd.concat(parts, ignore_index=True)
              .merge(actual, on=keys + ["target_month"], how="left")
              .merge(last, on=keys + ["cutoff_month"], how="left"))
    tested[["target", "baseline"]] = tested[["target", "baseline"]].fillna(0.0)
    return tested


def backtest(forecaster, daily, clients, cutoff):
    """Train the forecaster without the last 6 months (the data known at the end of the first test cut-off), then
    test it on them (test_forecasts). Returns the test rows; the forecaster is now the backtest forecaster."""
    first = add_months(cutoff, -HOLDOUT_MONTHS)
    forecaster.fit(until(daily, first), clients, first)
    return test_forecasts(forecaster, daily, clients, cutoff)


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def test_results(tested):
    """Test scores per months ahead (1, 2, 3) and overall (months_ahead ALL), for the TOTAL rows (level TOTAL) and the
    category rows (level CATEGORIES). The overall scores, which promotion uses, are on all rows."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    level = np.where(tested["category"] == "TOTAL", "TOTAL", "CATEGORIES")
    results = pd.concat([
        tested.assign(level=level).groupby(["level", "months_ahead"])[scored].apply(scores).reset_index(),
        tested.assign(level=level).groupby("level")[scored].apply(scores).reset_index().assign(months_ahead="ALL"),
        scores(tested).to_frame().T.assign(level="ALL", months_ahead="ALL")], ignore_index=True)
    results["months_ahead"] = results["months_ahead"].astype(str)
    return results[["level", "months_ahead"] + [c for c in results if c not in ("level", "months_ahead")]]


def run_metrics(results):
    """The scores recorded in Experiment Tracking: overall, and the EV error of the TOTAL rows per months ahead.
    results: from test_results."""
    overall = results[(results["level"] == "ALL")].iloc[0]
    metrics = {k: overall[c] for k, c in (("wape", "wape_model"), ("wape_baseline", "wape_baseline"),
                                          ("coverage_p10_p90", "coverage_p10_p90"), ("below_p10", "below_p10"),
                                          ("above_p90", "above_p90"))}
    for row in results[(results["level"] == "TOTAL") & (results["months_ahead"] != "ALL")].itertuples():
        metrics[f"wape_total_{row.months_ahead}m"] = row.wape_model
    return metrics


def _lab_test():
    """The lab's data and the production model's test, the same as champion / challenger's: daily flows, clients,
    the production model's ID and its backtest forecaster's test rows (None without a production model)."""
    import tracking
    from cashflow import read_dataset
    daily, clients = read_dataset("daily_flows"), read_dataset("clients")
    production_id = tracking.production_model_id()
    production = (test_forecasts(tracking.load_backtest(production_id), daily, clients, cutoff_month())
                  if production_id else None)
    return daily, clients, production_id, production


def _by_months_ahead(tested):
    """Overall test scores per months ahead (all rows), indexed by months ahead (1, 2, 3, ALL)."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    per = tested.groupby("months_ahead")[scored].apply(scores)
    per.index = per.index.astype(str)
    per.loc["ALL"] = scores(tested)
    return per


def _record_try(forecaster, tested, production_id):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab). Returns the run ID."""
    import tracking
    params = {"forecaster": forecaster.name, **getattr(forecaster, "params", {}),
              "cutoff_month": f"{cutoff_month():%Y-%m}", "test_rows": len(tested),
              "compared_with": production_id or "none"}
    run_name = f"{forecaster.name}_{pd.Timestamp.now():%Y%m%dT%H%M}"
    return tracking.log_lab_run(run_name, params, run_metrics(test_results(tested)), forecaster)


def lab_try(forecaster, record=True):
    """Try a forecaster against the production model without touching production (the lab: recipe try_forecaster).
    It is trained without the last 6 months and tested on them, the way champion / challenger tests a new model, and
    the try is recorded in Experiment Tracking (experiment cashflow_lab, never deployed).
    Returns: per months ahead (1, 2, 3, ALL) the EV error of the simple guess, of the production model and of the
    candidate, and their P10-P90 coverage; the decision champion / challenger would take and its reason; the lab run
    ID (None if not recorded)."""
    import tracking
    daily, clients, production_id, production = _lab_test()
    candidate = backtest(forecaster, daily, clients, cutoff_month())
    mine = _by_months_ahead(candidate)
    table = pd.DataFrame({"simple_guess_error": mine["wape_baseline"], "production_error": np.nan,
                          "candidate_error": mine["wape_model"], "production_coverage": np.nan,
                          "candidate_coverage": mine["coverage_p10_p90"]})
    if production is not None:
        theirs = _by_months_ahead(production)
        table["production_error"], table["production_coverage"] = theirs["wape_model"], theirs["coverage_p10_p90"]
    decision, reason = tracking.decide(scores(candidate), scores(production) if production is not None else None)
    run_id = _record_try(forecaster, candidate, production_id) if record else None
    return table.rename_axis("months_ahead"), decision, reason, run_id


def lab_compare(forecasters, record=True):
    """Try several forecasters at once against the production model, on the same test as lab_try and champion /
    challenger, without touching production. Each forecaster is trained without the last 6 months; each try is
    recorded in Experiment Tracking (experiment cashflow_lab) unless record is False. Returns a dict:
        board       one row per model (production first, simple guess last): EV error, P10-P90 coverage, share below
                    P10 and above P90, the decision champion / challenger would take and its reason, the lab run ID
        errors      EV error per months ahead: the simple guess, the production model and each forecaster
        coverage    P10-P90 coverage per months ahead: the production model and each forecaster
        tested      the test rows with each model's p10, ev, p90, by name, for your own analysis"""
    import tracking
    names = [f.name for f in forecasters]
    if len(set(names)) != len(names) or {"production", "simple_guess_baseline"} & set(names):
        raise ValueError(f"Give each forecaster its own name: {names}")
    daily, clients, production_id, production = _lab_test()
    tested = {} if production is None else {"production": production}
    for forecaster in forecasters:
        tested[forecaster.name] = backtest(forecaster, daily, clients, cutoff_month())
    run_ids = {f.name: _record_try(f, tested[f.name], production_id) for f in forecasters} if record else {}
    reference = scores(production) if production is not None else None
    board = []
    for name, rows in tested.items():
        s = scores(rows)
        decision, reason = (("", f"the production model, {production_id}") if name == "production"
                            else tracking.decide(s, reference))
        board.append({"model": name, "ev_error": s["wape_model"], "coverage_p10_p90": s["coverage_p10_p90"],
                      "below_p10": s["below_p10"], "above_p90": s["above_p90"], "decision": decision,
                      "reason": reason, "lab_run_id": run_ids.get(name, "")})
    board.append({"model": "simple_guess_baseline", "ev_error": scores(next(iter(tested.values())))["wape_baseline"],
                  "decision": "", "reason": "the same as the month before: the bar to beat", "lab_run_id": ""})
    board = pd.DataFrame(board).set_index("model")
    per = {name: _by_months_ahead(rows) for name, rows in tested.items()}
    first = next(iter(per.values()))
    errors = pd.DataFrame({"simple_guess_baseline": first["wape_baseline"],
                           **{name: found["wape_model"] for name, found in per.items()}})
    coverage = pd.DataFrame({name: found["coverage_p10_p90"] for name, found in per.items()})
    return {"board": board, "errors": errors, "coverage": coverage, "tested": tested}
'''

In [ ]:
TRACKING_CODE = r'''
"""Model tracking and promotion (project library: python/tracking.py), for training and the lab.

Every training run is one model with its own model ID, recorded in Dataiku's Experiment Tracking (MLflow; artifacts in
the managed folder "experiments"): its forecaster and settings, the cut-off, test scores on the last 6 months, the
backtest forecaster (trained without those months, for champion / challenger tests) and one MLflow model (the final
forecaster, which returns P10, EV and P90). It is deployed as a new version (version ID = model ID) of the saved model
cash_forecast (interface 3, cashflow.SAVED_MODEL). The production model is the active version; promoting a model
activates it, and the nightly forecast always uses the active version.
Lab tries (notebook forecast_lab) go to their own experiment, cashflow_lab, in the folder lab_experiments. They are
never deployed.
"""
import inspect
import os
import pickle
import sys
import tempfile

import pandas as pd

from cashflow import SAVED_MODEL

EXPERIMENT, FOLDER = "cashflow_forecast", "experiments"
LAB_EXPERIMENT, LAB_FOLDER = "cashflow_lab", "lab_experiments"
# The promotion rules: project variables (change them in the project's variables, no code change), with these
# defaults. A candidate is promoted only if all hold:
PROMOTION_RULES = {
    "promotion_min_coverage": 0.70,      # its P10-P90 coverage is at least this...
    "promotion_max_coverage": 0.90,      # ...and at most this (80% is the aim)
    "promotion_beat_simple_guess": True,  # its EV error is lower than the simple guess's (same as the month before)
    "promotion_min_gain": 0.0,           # its EV error is lower than the production model's by more than this share
                                         # (0.02: at least 2% lower). A tie keeps the production model.
}


def _project():
    import dataiku
    return dataiku.api_client().get_default_project()


def _mlflow(folder):
    """Experiment Tracking in the folder. By its ID: outside a recipe (the lab notebook), setup_mlflow does not resolve
    a folder name."""
    import dataiku
    return _project().setup_mlflow(managed_folder=dataiku.Folder(folder).get_id())


def _experiment_name(mlflow, experiment, folder):
    """The experiment, kept in the folder. If an experiment of that name points to another folder (the folder was
    deleted and created again, for example when the flow was deleted by hand), its files are gone: use
    <experiment>_<folder id> instead."""
    import dataiku
    folder_id = dataiku.Folder(folder).get_id()
    found = mlflow.get_experiment_by_name(experiment)
    if found is None or f"//{folder_id}/" in (found.artifact_location or ""):
        return experiment
    return f"{experiment}_{folder_id}"


def new_model_id(cutoff):
    """CFF_<cut-off month>_<training time>, for example CFF_202610_20261104T1215 (also the saved model version)."""
    return f"CFF_{cutoff:%Y%m}_{pd.Timestamp.now():%Y%m%dT%H%M}"


def promotion_rules():
    """The promotion rules: the project variables PROMOTION_RULES names, or their defaults."""
    import dataiku
    variables = dataiku.get_custom_variables()
    rules = {}
    for name, default in PROMOTION_RULES.items():
        value = variables.get(name, default)
        rules[name] = (str(value).strip().lower() in ("true", "1", "yes")) if isinstance(default, bool) else float(value)
    return rules


def saved_model(name=SAVED_MODEL):
    project = _project()
    return project.get_saved_model(next(m["id"] for m in project.list_saved_models() if m["name"] == name))


def forecast_model(forecaster):
    """An MLflow model around a trained forecaster. The nightly forecast unwraps it and calls the forecaster
    (forecast.production_forecaster); called as an MLflow model, it takes daily flows (interface 1) and returns the
    forecast of the month after the last day it was given, without client details."""
    import mlflow.pyfunc
    from cashflow import month_start
    from models import forecast

    class ForecastModel(mlflow.pyfunc.PythonModel):
        def __init__(self, forecaster):
            self.forecaster = forecaster

        def predict(self, context, model_input, params=None):
            cutoff = month_start(model_input["day"]).max()
            return forecast(self.forecaster, model_input, None, cutoff)

    return ForecastModel(forecaster)


def _carry_code(forecaster):
    """The models carry their own copy of the library code (pickled by value: cashflow, models, tracking and the
    forecaster's own module, forecasters.py for example), so they keep working when the library changes."""
    import cloudpickle
    import cashflow
    import models
    own = sys.modules.get(type(forecaster).__module__)
    for module in (cashflow, models, sys.modules[__name__], own):
        if module is not None and module.__name__ != "__main__":
            cloudpickle.register_pickle_by_value(module)


def _record(mlflow, params, metrics, backtest):
    """A run's settings, test scores and backtest forecaster."""
    import cloudpickle
    mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
    mlflow.log_metrics({k: float(v) for k, v in metrics.items() if pd.notna(v)})
    with tempfile.TemporaryDirectory() as tmp:
        with open(os.path.join(tmp, "forecaster.pkl"), "wb") as f:
            cloudpickle.dump(backtest, f)
        mlflow.log_artifacts(tmp, artifact_path="backtest")


def log_model(model_id, params, metrics, backtest, final):
    """Record one training run in Experiment Tracking, then deploy its final forecaster as version model_id of the
    saved model cash_forecast (not activated: the promotion does that)."""
    _carry_code(final)
    with _mlflow(FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, EXPERIMENT, FOLDER))
        with mlflow.start_run(run_name=model_id) as run:
            mlflow.set_tag("model_id", model_id)
            _record(mlflow, params, metrics, backtest)
            where = "name" if int(mlflow.__version__.split(".")[0]) >= 3 else "artifact_path"   # MLflow 3 or 2
            mlflow.pyfunc.log_model(python_model=forecast_model(final), **{where: "forecaster"})
            run_id = run.info.run_id
    extension = _project().get_mlflow_extension()
    code_env = _project().get_recipe("train_and_test").get_settings().get_recipe_params()["envSelection"]["envName"]
    extension.set_run_inference_info(run_id=run_id, prediction_type="REGRESSION", code_env_name=code_env)
    # which model of the run: model_name in DSS 15 (MLflow 3 models), model_sub_folder in DSS 14 (the artifact path)
    which = "model_name" if "model_name" in inspect.signature(extension.deploy_run_model).parameters else "model_sub_folder"
    model = saved_model()
    before = (model.get_active_version() or {}).get("id")
    extension.deploy_run_model(run_id, model.id, version_id=model_id, activate=False, skip_expensive_reports=True,
                               **{which: "forecaster"})
    if before and (model.get_active_version() or {}).get("id") != before:
        model.set_active_version(before)       # DSS 14 activates every deployed version: only a promotion may


def log_lab_run(run_name, params, metrics, backtest):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab, folder lab_experiments): its settings, test
    scores and forecaster. Never deployed. Returns the run ID."""
    _carry_code(backtest)
    with _mlflow(LAB_FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, LAB_EXPERIMENT, LAB_FOLDER))
        with mlflow.start_run(run_name=run_name) as run:
            _record(mlflow, params, metrics, backtest)
            return run.info.run_id


def runs():
    """Every training run, newest first: model_id, run_id, trained_at, its settings (params.*) and metrics."""
    with _mlflow(FOLDER) as mlflow:
        # sorted here: DSS 14 cannot sort runs by start time in the search
        found = mlflow.search_runs(experiment_names=[_experiment_name(mlflow, EXPERIMENT, FOLDER)])
    if found.empty:
        return pd.DataFrame(columns=["model_id", "run_id", "trained_at"])
    found = found.sort_values("start_time", ascending=False).reset_index(drop=True)
    return found.rename(columns={"tags.model_id": "model_id", "start_time": "trained_at"})


def latest_model_id():
    """The most recently trained model."""
    found = runs()
    if found.empty:
        raise ValueError("No model trained yet: run the scenario Retrain, test and promote")
    return found.iloc[0]["model_id"]


def production_model_id():
    """The production model: the active version of the saved models, if it was ever promoted (a version Dataiku
    activated on its own when the first one was deployed does not count until the promotion)."""
    model_id = (saved_model().get_active_version() or {}).get("id")
    return model_id if model_id in promoted_model_ids() else ""


def promoted_model_ids():
    """Every model ID promoted so far (read through the API: also works in a recipe that writes promotion_log)."""
    try:
        log = _project().get_dataset("promotion_log")
        columns = [c["name"] for c in log.get_schema()["columns"]]
        model, decision = columns.index("candidate_model_id"), columns.index("decision")
        return {row[model] for row in log.iter_rows() if row[decision] == "PROMOTE"}
    except Exception:            # no promotion yet
        return set()


def load_backtest(model_id):
    """The backtest forecaster of a training run (trained without the last 6 months, for champion / challenger)."""
    found = runs()
    match = found[found["model_id"] == model_id]
    if match.empty:
        raise ValueError(f"Model {model_id} not found in Experiment Tracking ({EXPERIMENT})")
    with _mlflow(FOLDER) as mlflow:
        path = mlflow.artifacts.download_artifacts(run_id=match.iloc[0]["run_id"], artifact_path="backtest")
    with open(os.path.join(path, "forecaster.pkl"), "rb") as f:
        return pickle.load(f)


def decide(candidate, production, rules=None):
    """Champion / challenger: the decision (PROMOTE or KEEP) and its reason, from the test scores of the candidate and
    of the production model (models.scores), with the promotion rules (promotion_rules: project variables). production
    None: no production model to compare with, the candidate is promoted (the forecast needs one) with its failed
    checks as a warning."""
    rules = rules or promotion_rules()
    low, high = rules["promotion_min_coverage"], rules["promotion_max_coverage"]
    checks = []
    if not low <= candidate["coverage_p10_p90"] <= high:
        checks.append(f"coverage {candidate['coverage_p10_p90']:.1%} outside {low:.0%}-{high:.0%}")
    if rules["promotion_beat_simple_guess"] and candidate["wape_model"] >= candidate["wape_baseline"]:
        checks.append(f"does not beat the simple guess ({candidate['wape_model']:.1%} vs "
                      f"{candidate['wape_baseline']:.1%})")
    if production is None:
        return "PROMOTE", (f"no production model to compare with, EV error {candidate['wape_model']:.2%}, coverage "
                           f"{candidate['coverage_p10_p90']:.1%}" + (f" (warning: {'; '.join(checks)})" if checks else ""))
    if checks:
        return "KEEP", f"candidate {checks[0]}"
    needed = production["wape_model"] * (1 - rules["promotion_min_gain"])
    if candidate["wape_model"] >= needed:
        gain = f" by more than {rules['promotion_min_gain']:.0%}" if rules["promotion_min_gain"] else ""
        return "KEEP", (f"candidate EV error {candidate['wape_model']:.2%} not lower than production "
                        f"{production['wape_model']:.2%}{gain}")
    return "PROMOTE", (f"candidate EV error {candidate['wape_model']:.2%} vs production {production['wape_model']:.2%}, "
                       f"coverage {candidate['coverage_p10_p90']:.1%}")


def apply_promotion():
    """Apply the latest decision in promotion_log: on PROMOTE, activate the candidate's version of the saved model.
    Run by a scenario step after the champion / challenger test."""
    import dataiku
    log = dataiku.Dataset("promotion_log").get_dataframe()
    last = log.sort_values("decided_at").iloc[-1]
    if last["decision"] == "PROMOTE":
        saved_model().set_active_version(last["candidate_model_id"])
        print(f"Promoted {last['candidate_model_id']}: {last['reason']}")
    else:
        print(f"Kept the production model: {last['reason']}")
    return production_model_id()
'''

In [ ]:
FORECAST_CODE = r'''
"""Inference (project library: python/forecast.py): the production model and the final forecast table. Inference reads
nothing from training except the active version of the saved model cash_forecast (interface 3, cashflow.SAVED_MODEL).
"""
import numpy as np
import pandas as pd

from cashflow import MIN_HISTORY_MONTHS, SAVED_MODEL, add_months


def _saved_model_id():
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {m["name"]: m["id"] for m in project.list_saved_models()}
    if SAVED_MODEL not in ids:
        raise ValueError(f"Saved model {SAVED_MODEL} not found. Run the setup notebook.")
    return ids[SAVED_MODEL]


def active_model_id():
    """The production model: the active version of the saved model cash_forecast."""
    import dataiku
    model = dataiku.api_client().get_default_project().get_saved_model(_saved_model_id())
    model_id = (model.get_active_version() or {}).get("id")
    if not model_id:
        raise ValueError("No production model yet: run the scenario Retrain, test and promote")
    return model_id


def production_forecaster():
    """The production model's trained forecaster (the active version of cash_forecast), and its model ID. The recipe
    that calls it must have the saved model as an input."""
    import dataiku
    model_id = active_model_id()
    predictor = dataiku.Model(_saved_model_id()).get_predictor(model_id)
    return predictor.model.unwrap_python_model().forecaster, model_id


def forecast_table(pred, history, last_month):
    """The forecast: one row per client, category (and TOTAL) and month ahead, the net cash flow (money in minus money
    out) as P10, EV and P90 in SAR. pred: the production model's forecast (models.forecast); history: months of
    history per client (cashflow.history_months). A client with less than MIN_HISTORY_MONTHS months of history gets
    no values (status NOT_ENOUGH_HISTORY)."""
    frame = pred.merge(history.reset_index(), on="client_no", how="left")
    enough = frame["history_months"].fillna(0) >= MIN_HISTORY_MONTHS
    frame.loc[~enough, ["p10", "ev", "p90"]] = np.nan
    frame["cutoff_month"] = last_month
    frame["target_month"] = frame["months_ahead"].map({h: add_months(last_month, h) for h in (1, 2, 3)})
    frame["status"] = np.where(enough, "OK", "NOT_ENOUGH_HISTORY")
    return frame.sort_values(["client_no", "category", "months_ahead"]).reset_index(drop=True)
'''

In [ ]:
FORECASTERS_CODE = r'''
"""The data scientist's forecasters (project library: python/forecasters.py). The setup notebook creates this file once
and never overwrites it: it is yours.

Try a forecaster first: in the notebook forecast_lab, or set the project variable lab_forecaster to its name and
build lab_results (Lab zone). To put one in production:
    1. add its class here and to FORECASTERS below,
    2. set the project variable forecaster to its name,
    3. run the scenario Retrain, test and promote: champion / challenger promotes it only if it passes the promotion
       rules (project variables promotion_*) on the same last 6 months. To go back, set forecaster to recent_average.

A forecaster is a class with (details in models.py, and RecentAverage there):
    name            the value of the project variable forecaster that selects it
    params          its settings (a dict), recorded in Experiment Tracking
    fit(daily, clients, cutoff)
                    learns from the daily flows (client_no, day, category, money_in, money_out, postings) up to the
                    end of the cut-off month, and the clients
    predict(daily, clients, cutoff)
                    one row per client, category (and TOTAL) and months ahead (1, 2, 3): client_no, category,
                    months_ahead, p10, ev, p90 (the net cash flow of that month, SAR)
cashflow.monthly_matrix(daily, cutoff) gives the net cash flow per client and category (and TOTAL) and month.
"""
from models import RecentAverage


class RecentAverage6(RecentAverage):
    """Example: the placeholder with the average of the last 6 months instead of 3."""
    name = "recent_average_6m"
    params = {**RecentAverage.params, "months": 6}


FORECASTERS = [RecentAverage6]
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["stage_tables"] = r'''
# Staging: the 2 warehouse tables, read-only, turned into the project's tables, all in Dataiku. This is interface 1,
# what the data scientist's models get:
#   clients: the business clients of DIM_CUSTOMER (companies and sole proprietors), standard names and values. Read
#   whole every run (it is small).
#   daily_flows: the postings of FCT_STATEMENT_ENTRY summed per client, day and category (money in, money out):
#   business clients only, no reversals, no transfers between the client's own accounts. Only the NEW days are read
#   and added: the days after the last day already staged, up to the end of the cut-off month. When nothing is
#   staged yet (the first run, or after the scenario Full rebuild), the whole history is read.
#   staging_log: one row per run (appended): what was read, and the last day now in daily_flows.
# A big read never sits in memory whole: the postings are spread over files on disk by TRNS_RFR (both sides of a
# reversal or of an own-account transfer in the same file) and summed one file at a time; the totals are then spread
# by client and added up one file at a time.
import json
import os
import tempfile
import time

import dataiku
import pandas as pd

from cashflow import Spread, cutoff_month, iter_dataset
from warehouse import (COLUMNS, TABLES, business_clients, daily_flows, to_standard, transactions_query,
                       value_day)

CHUNK, FILES = 200_000, 128
KEYS = ["client_no", "day", "category"]
EMPTY = pd.DataFrame({"client_no": pd.Series(dtype=str), "day": pd.Series(dtype="datetime64[ns]"),
                      "category": pd.Series(dtype=str), "money_in": pd.Series(dtype=float),
                      "money_out": pd.Series(dtype=float), "postings": pd.Series(dtype="int64")})
started = time.time()
cutoff = cutoff_month()
end = cutoff + pd.DateOffset(months=1)             # postings up to the end of the cut-off month


def read_table(table, since=None):
    """The table's chunks with the standard names, only the columns the project uses. Postings: only the value dates
    from since (None: all) to the end of the cut-off month; through the database connection the warehouse filters
    them (transactions_query), from files they are filtered here."""
    dataset = dataiku.Dataset(TABLES[table][0])
    if table == "transactions":
        try:
            name = dataset.get_location_info()["info"].get("quotedResolvedTableName")
        except Exception:
            name = None
        if name:
            reader = dataiku.SQLExecutor2(dataset=dataset).query_to_iter(transactions_query(name, since, end))
            names, rows = [c["name"] for c in reader.get_schema()], []
            for row in reader.iter_tuples():
                rows.append(row)
                if len(rows) == CHUNK:
                    yield to_standard(pd.DataFrame(rows, columns=names), table)
                    rows = []
            if rows:
                yield to_standard(pd.DataFrame(rows, columns=names), table)
            return
    for chunk in dataset.iter_dataframes(chunksize=CHUNK, columns=list(COLUMNS[table].values())):
        chunk = to_standard(chunk, table)
        if table == "transactions":
            day = value_day(chunk["value_date"])
            chunk = chunk[(day < end) & ((day >= since) if since is not None else True)]
        yield chunk


# what is already staged: staging_log (this recipe adds to it, so it reads it outside the flow's inputs)
try:
    log = dataiku.Dataset("staging_log", ignore_flow=True).get_dataframe()
except Exception:
    log = pd.DataFrame()
staged = {}                                        # postings per month already in daily_flows
for months in (log["postings_per_month"] if len(log) else []):
    for month, n in json.loads(months).items():
        staged[month] = staged.get(month, 0) + n
last_day = pd.to_datetime(log["last_day"]).max() if len(log) else None
if last_day is None:
    try:
        first = next(iter_dataset("daily_flows", rows=1, ignore_flow=True), EMPTY)
    except Exception:
        first = EMPTY
    if len(first):
        raise ValueError("staging_log is empty but daily_flows is not: run the scenario Full rebuild")
since = None if last_day is None else last_day + pd.Timedelta(days=1)
mode = "FULL" if since is None else "NOTHING_NEW" if since >= end else "NEW_DAYS"

clients = business_clients(pd.concat([business_clients(chunk) for chunk in read_table("clients")],
                                     ignore_index=True))
if clients.empty:
    raise ValueError("No business client in DIM_CUSTOMER: check the codes RETAIL and BUSINESS_ID in warehouse.py")
numbers = set(clients["client_no"])
dataiku.Dataset("clients").write_with_schema(clients)

read_rows = kept = added = 0
new = {}                                           # postings per month read now
out = dataiku.Dataset("daily_flows")
out.write_schema_from_dataframe(EMPTY)
with tempfile.TemporaryDirectory() as spread:
    if mode != "NOTHING_NEW":
        postings = Spread(os.path.join(spread, "postings"), FILES)
        for chunk in read_table("transactions", since):
            read_rows += len(chunk)
            chunk = chunk[chunk["client_no"].isin(numbers)]
            kept += len(chunk)
            if len(chunk):
                last_day = max(value_day(chunk["value_date"]).max(), last_day or pd.Timestamp.min)
                postings.add(chunk, chunk["txn_ref"].fillna(chunk["entry_id"].astype(str)))
        totals = Spread(os.path.join(spread, "totals"), FILES)
        for part in postings.parts():
            flows = daily_flows(part, numbers, cutoff)
            for month, n in flows.groupby(flows["day"].dt.strftime("%Y-%m"))["postings"].sum().items():
                new[month] = new.get(month, 0) + int(n)
            totals.add(flows, "client_no")

    # the postings must reach the cut-off month: none, or far fewer than the month before, means a late load. Checked
    # before anything is added, so a failed run adds nothing and the next run reads the same days again.
    months = {m: staged.get(m, 0) + new.get(m, 0) for m in set(staged) | set(new)}
    last, before = months.get(f"{cutoff:%Y-%m}", 0), months.get(f"{cutoff - pd.DateOffset(months=1):%Y-%m}", 0)
    if last == 0 or last < 0.5 * before:
        raise ValueError(f"FCT_STATEMENT_ENTRY has {last:,} postings of business clients in {cutoff:%Y-%m} "
                         f"({before:,} the month before): the warehouse load for that month looks late or partial")

    if mode != "NOTHING_NEW":
        with out.get_writer() as writer:
            for part in totals.parts():
                part = (part.groupby(KEYS, as_index=False)[["money_in", "money_out", "postings"]].sum()
                        .round({"money_in": 2, "money_out": 2}))
                writer.write_dataframe(part[list(EMPTY.columns)])
                added += len(part)

dataiku.Dataset("staging_log").write_with_schema(pd.DataFrame([{
    "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "mode": mode,
    "cutoff_month": f"{cutoff:%Y-%m}",
    "read_from": f"{since:%Y-%m-%d}" if mode == "NEW_DAYS" else "",
    "read_to": f"{end - pd.Timedelta(days=1):%Y-%m-%d}",
    "postings_read": read_rows,
    "postings_kept": kept,
    "daily_rows_added": added,
    "last_day": f"{last_day:%Y-%m-%d}",
    "postings_per_month": json.dumps(dict(sorted(new.items()))),
    "clients": len(clients),
    "seconds": round(time.time() - started, 1)}]))
print(f"{mode}: {len(clients):,} business clients; {read_rows:,} postings read"
      + (f" from {since:%Y-%m-%d}" if since is not None else " (whole history)")
      + f", {kept:,} kept; {added:,} daily totals added; last day staged {last_day:%Y-%m-%d}")
'''

In [ ]:
RECIPES["train_and_test"] = r'''
# Train and test one model, with its own model ID, tracked in Experiment Tracking with its forecaster, settings and
# scores. The forecaster is the one the project variable forecaster names (default recent_average, in models.py; the
# data scientist's own are in forecasters.py). It learns from the daily flows and the clients (interface 1).
#   backtest: trained without the last 6 months, then tested on them (models.backtest, test_results)
#   final: trained on everything up to the cut-off, deployed as version <model ID> of the saved model cash_forecast
#   (not active: the promotion activates it)
import dataiku

from cashflow import cutoff_month, read_dataset, until
from models import backtest, new_forecaster, run_metrics, test_results
from tracking import log_model, new_model_id

cutoff = cutoff_month()
daily, clients = until(read_dataset("daily_flows"), cutoff), read_dataset("clients")
backtested, final = new_forecaster(), new_forecaster()
tested = backtest(backtested, daily, clients, cutoff)
final.fit(daily, clients, cutoff)
model_id = new_model_id(cutoff)

results = test_results(tested)
results.insert(0, "model_id", model_id)
metrics = run_metrics(results)
params = {"forecaster": final.name, **getattr(final, "params", {}), "cutoff_month": f"{cutoff:%Y-%m}",
          "load_id": dataiku.get_custom_variables().get("load_id", ""), "daily_rows": len(daily),
          "clients": daily["client_no"].nunique(), "test_rows": len(tested)}
log_model(model_id, params, metrics, backtested, final)
print(f"Model {model_id} ({final.name}): EV error {metrics['wape']:.1%} (simple guess {metrics['wape_baseline']:.1%}), "
      f"coverage {metrics['coverage_p10_p90']:.1%}")

dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["champion_challenger"] = r'''
# Champion / challenger: the latest trained model (candidate) against the production model, both tested on the
# same last 6 months with their backtest forecasters (trained without those months). The rules are project variables
# (tracking.PROMOTION_RULES, applied by tracking.decide): promoted if its P10-P90 coverage is in the range
# (default 70%-90%), it beats the simple guess (the same as the month before) and its EV error is lower than the
# production model's (by more than promotion_min_gain; a tie keeps the production model). If there is no production
# model to compare with (none yet, or its tracked files were deleted), the candidate is promoted (the forecast needs
# one), with its failed checks as a warning. The production model is the active version of the saved model
# cash_forecast. The decision is added to promotion_log; the next scenario step applies it (activates the
# candidate's version).
import dataiku
import pandas as pd

from cashflow import cutoff_month, read_dataset, until
from models import scores, test_forecasts
from tracking import decide, latest_model_id, load_backtest, production_model_id, promotion_rules, runs

cutoff = cutoff_month()
daily, clients = until(read_dataset("daily_flows"), cutoff), read_dataset("clients")
candidate = latest_model_id()
production = production_model_id()
tracked = runs()
if production and production not in set(tracked["model_id"]):     # its tracked files were deleted: nothing to test
    print(f"Production model {production} is not in Experiment Tracking: the candidate is judged on its own")
    production = ""
tested = {m: scores(test_forecasts(load_backtest(m), daily, clients, cutoff)) for m in {candidate, production} if m}
c = tested[candidate]
p = tested.get(production) if production and production != candidate else None
if production == candidate:
    decision, reason = "KEEP", "the candidate is already the production model"
else:
    decision, reason = decide(c, p, promotion_rules())
print(decision, "-", reason)

forecaster = dict(zip(tracked["model_id"], tracked["params.forecaster"])) if "params.forecaster" in tracked else {}
dataiku.Dataset("promotion_log").write_with_schema(pd.DataFrame([{
    "decided_at": pd.Timestamp.now().floor("s"), "cutoff_month": dataiku.get_custom_variables()["cutoff_month"],
    "candidate_model_id": candidate, "candidate_forecaster": forecaster.get(candidate),
    "production_model_id": production or None, "production_forecaster": forecaster.get(production),
    "candidate_wape": c["wape_model"], "production_wape": p["wape_model"] if p is not None else None,
    "baseline_wape": c["wape_baseline"], "candidate_coverage": c["coverage_p10_p90"],
    "production_coverage": p["coverage_p10_p90"] if p is not None else None,
    "decision": decision, "reason": reason, "rules": str(promotion_rules())}]))
'''

In [ ]:
RECIPES["final_forecast"] = r'''
# Final forecast: the production model (the active version of the saved model cash_forecast) forecasts, for each
# active business client, the net cash flow (money in minus money out) of each of the next 3 months, per category and
# in TOTAL, as P10, EV and P90 in SAR, from the client's daily flows (interface 1). Every row says which model
# (model_id: the active version) and which data (load_id) produced it. Inference reads nothing else from training.
# Built to scale: the daily flows are read in chunks and spread over files on disk by client, then forecast one file
# at a time (about inference_clients_per_chunk clients each, project variable), so memory holds one chunk of clients
# whatever their number. Each chunk's rows are written at once to:
#   final_forecast: the forecast
#   forecast_archive: one compressed CSV file per cut-off month and model (forecast_<cut-off>_<model ID>.csv.gz)
#   inference_log: one row per run (appended): which model and load, what it produced, how long each step took, and
#   the most memory the run used
import gzip
import math
import resource
import tempfile
import time

import dataiku
import pandas as pd

from cashflow import Spread, cutoff_month, iter_dataset, read_dataset, history_months, until
from forecast import forecast_table, production_forecaster
from models import active, forecast

COLUMNS = ["client_no", "category", "cutoff_month", "months_ahead", "target_month", "p10", "ev", "p90",
           "history_months", "status", "model_id", "load_id"]
started = time.time()
variables = dataiku.get_custom_variables()
last_month = cutoff_month()
forecaster, model_id = production_forecaster()
_, clients = active(pd.DataFrame(columns=["client_no"]), read_dataset("clients"))     # the active clients
numbers = set(clients["client_no"])
per_chunk = int(variables.get("inference_clients_per_chunk") or 5000)
chunks = max(1, math.ceil(len(numbers) / per_chunk))
name = f"forecast_{last_month:%Y-%m}_{model_id}.csv.gz"

with tempfile.TemporaryDirectory() as folder:
    spread = Spread(folder, chunks)
    daily_rows = 0
    for daily in iter_dataset("daily_flows"):
        daily = until(daily, last_month)
        daily = daily[daily["client_no"].isin(numbers)]
        daily_rows += len(daily)
        spread.add(daily, "client_no")
    read_seconds = time.time() - started

    out = dataiku.Dataset("final_forecast")
    writer, rows, without_history, forecast_clients = None, 0, 0, 0
    with dataiku.Folder("forecast_archive").get_writer(name) as archive_file, \
            gzip.GzipFile(fileobj=archive_file, mode="wb") as archive:
        for daily in spread.parts():
            group = clients[clients["client_no"].isin(set(daily["client_no"]))]
            pred = forecast(forecaster, daily, group, last_month)
            frame = forecast_table(pred, history_months(daily, last_month), last_month)
            frame["model_id"] = model_id
            frame["load_id"] = variables.get("load_id", "")
            frame = frame[COLUMNS].round(2)
            if writer is None:
                out.write_schema_from_dataframe(frame)
                writer = out.get_writer()
            writer.write_dataframe(frame)
            archive.write(frame.to_csv(index=False, header=rows == 0).encode())
            rows += len(frame)
            first = frame.drop_duplicates("client_no")
            forecast_clients += len(first)
            without_history += int((first["status"] != "OK").sum())
    if writer is None:
        raise ValueError("No active client with daily flows to forecast: check daily_flows and clients")
    writer.close()

total_seconds = time.time() - started
dataiku.Dataset("inference_log").write_with_schema(pd.DataFrame([{
    "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "load_id": variables.get("load_id", ""),
    "as_of_date": variables.get("as_of_date", ""),
    "cutoff_month": f"{last_month:%Y-%m}",
    "model_id": model_id,
    "clients": forecast_clients,
    "clients_not_enough_history": without_history,
    "forecast_rows": rows,
    "daily_rows_read": daily_rows,
    "chunks": chunks,
    "seconds_reading": round(read_seconds, 1),
    "seconds_forecasting": round(total_seconds - read_seconds, 1),
    "seconds_total": round(total_seconds, 1),
    "peak_memory_mb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024),
    "archive_file": name}]))
print(f"Forecast with {model_id}: {forecast_clients:,} clients in {chunks} chunks, {rows:,} rows "
      f"({without_history:,} clients without enough history), {total_seconds:.0f} s; archived {name}")
'''

In [ ]:
RECIPES["try_forecaster"] = r'''
# Lab (data scientist): try a forecaster against the production model without touching production. The forecaster is
# the one the project variable lab_forecaster names (models.py or forecasters.py). It is trained without the last 6
# months and tested on them, the way champion / challenger tests a new model. The try is recorded in Experiment
# Tracking (experiment cashflow_lab, folder lab_experiments) and added to lab_results: per months ahead, its EV error and
# P10-P90 coverage next to the production model's and the simple guess, and the decision champion / challenger would
# take. Nothing here changes the saved models. Run it by building lab_results (only built when asked by name).
import dataiku
import pandas as pd

from cashflow import cutoff_month
from models import lab_try, new_forecaster

forecaster = new_forecaster(dataiku.get_custom_variables().get("lab_forecaster") or "recent_average")
table, decision, reason, run_id = lab_try(forecaster)
print(f"{forecaster.name}: champion / challenger would decide {decision} ({reason})")

table = table.reset_index().assign(tried_at=pd.Timestamp.now().floor("s"), forecaster=forecaster.name,
                                   cutoff_month=f"{cutoff_month():%Y-%m}", decision=decision, reason=reason,
                                   lab_run_id=run_id)
cols = ["tried_at", "forecaster", "cutoff_month", "months_ahead", "simple_guess_error", "production_error",
        "candidate_error", "production_coverage", "candidate_coverage", "decision", "reason", "lab_run_id"]
dataiku.Dataset("lab_results").write_with_schema(table[cols])
'''

## 3. Build the flow

Uses the simulated warehouse tables, or links the warehouse's (read-only), and creates the Python recipes and their
CSV datasets, the folders `experiments` (tracked models), `forecast_archive` and `lab_experiments` (lab tries), the
saved model `cash_forecast`, and the project variables `forecaster` (`recent_average`), `lab_forecaster`
(`recent_average_6m`), `inference_clients_per_chunk` (5000) and the promotion rules (`promotion_*`) if they are not
set yet, in 5 zones. Training (folder
`experiments`), the promotion log and the lab are only rebuilt when asked by name, so the nightly batch never
retrains by accident.

In [ ]:
import json
import sys
import time
import types

import pandas as pd

# Safe to run again in the same project: what already exists is kept (datasets and their data, folders, the saved
# model and its versions, zones, the project variables, forecasters.py), and the code of the Python recipes and of the
# rest of the library is updated.
flow = project.get_flow()


def load_module(name, code):
    """A library file as a module of this notebook: the code it writes to the project library."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(code, module.__dict__)
    return module


cashflow = load_module("cashflow", LIBRARY_CODE)
warehouse = load_module("warehouse", WAREHOUSE_CODE)
tracking = load_module("tracking", TRACKING_CODE)
RAW = [warehouse_table for warehouse_table, _ in warehouse.TABLES.values()]
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}
COMPRESSED = ["daily_flows", "final_forecast"]       # the big ones: compressed CSV (gz)

# Python recipes: (inputs, outputs, flow zone). Every dataset is a CSV file (compressed for the big ones);
# experiments, forecast_archive and lab_experiments are folders; cash_forecast is the saved model.
# Each zone has an owner, and the zones only meet through three interfaces:
#   1. the data: daily_flows and clients (Staging) -> Training, Lab and Inference
#   2. the forecaster (models.py): lab -> training. The project variable forecaster names the one Retrain trains;
#      the Lab (try_forecaster, and the notebook forecast_lab) compares one with the production model, untouched.
#   3. the saved model's active version: training -> inference. Inference reads nothing else from training.
DATA = ["daily_flows", "clients"]
FLOW = {
    "stage_tables": (RAW, warehouse.STAGED, "Staging"),
    "train_and_test": (DATA, ["experiments", "test_results"], "Training"),
    "champion_challenger": (DATA + ["experiments"], ["promotion_log"], "Training"),
    "final_forecast": (DATA + [cashflow.SAVED_MODEL], ["final_forecast", "forecast_archive", "inference_log"],
                       "Inference"),
    "try_forecaster": (DATA + ["experiments"], ["lab_experiments", "lab_results"], "Lab"),
}
FOLDERS = ["experiments", "forecast_archive", "lab_experiments"]
# each run adds rows: the logs, and daily_flows (staging adds the new days)
APPENDED = ["daily_flows", "staging_log", "promotion_log", "inference_log", "lab_results"]
BUILT_BY_NAME = ["experiments", "promotion_log", "lab_experiments", "lab_results"]   # only built when asked by name

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

# the project library: shared (cashflow.py, warehouse.py), training and lab (models.py, tracking.py), inference
# (forecast.py). forecasters.py is the data scientist's: written only when it is created, never overwritten.
library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE), ("models.py", MODELS_CODE),
                        ("tracking.py", TRACKING_CODE), ("forecast.py", FORECAST_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)
if not python_folder.get_file("forecasters.py"):
    python_folder.add_file("forecasters.py").write(FORECASTERS_CODE)
    print("forecasters.py created (the data scientist's forecasters)")

# project variables, kept if they are already set: forecaster (the one Retrain trains), lab_forecaster (the one the
# recipe try_forecaster tries), inference_clients_per_chunk (how many clients final_forecast forecasts at a time)
variables = project.get_variables()
defaults = {"forecaster": "recent_average", "lab_forecaster": "recent_average_6m",
            "inference_clients_per_chunk": 5000, **tracking.PROMOTION_RULES}
missing = {k: v for k, v in defaults.items() if variables["standard"].get(k) in (None, "")}
if missing:
    variables["standard"].update(missing)
    project.set_variables(variables)
print("Forecasters: Retrain trains {forecaster}, the lab tries {lab_forecaster}; inference forecasts "
      "{inference_clients_per_chunk} clients at a time".format(**variables["standard"]))
print("Promotion rules:", {k: variables["standard"][k] for k in tracking.PROMOTION_RULES})


# what the project already has (read once, kept up to date below)
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}



def new_dataset(name):
    """Managed dataset of the project: a CSV file (comma-separated, with a header row) on CONNECTION. Kept if it
    exists."""
    if name in datasets:
        return name
    datasets.add(name)
    dataset = project.new_managed_dataset(name).with_store_into(CONNECTION).create()
    settings = dataset.get_settings()
    settings.get_raw()["formatType"] = "csv"
    settings.get_raw()["formatParams"].update(CSV, compress="gz" if name in COMPRESSED else "")
    if name in BUILT_BY_NAME:
        settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
    settings.save()
    return name


# the 2 warehouse tables: already here if the warehouse simulator wrote them (sandbox, CSV), otherwise read-only from
# the warehouse as they are (production): tables of WAREHOUSE_SQL_SCHEMA through the database connection
# WAREHOUSE_SQL_CONNECTION, or CSV files in the landing folder (LANDING_CONNECTION). Nothing is created there.
simulated = [name for name in RAW if name in datasets]
source = WAREHOUSE_SQL_CONNECTION or LANDING_CONNECTION
if WAREHOUSE_SQL_CONNECTION and len(simulated) < len(RAW):
    try:
        database = client.get_connection(WAREHOUSE_SQL_CONNECTION).get_info().get_type()   # Oracle, Teradata, ...
    except Exception as e:
        raise RuntimeError(f"Cannot use the connection {WAREHOUSE_SQL_CONNECTION!r} ({e}). Ask a Dataiku admin for a "
                           "read-only database connection to the warehouse, usable by your group, or check its "
                           "name in WAREHOUSE_SQL_CONNECTION.") from None
for name in RAW:
    if name in datasets:
        continue
    datasets.add(name)
    if WAREHOUSE_SQL_CONNECTION:
        dataset = project.create_sql_table_dataset(name, database, WAREHOUSE_SQL_CONNECTION, name,
                                                   WAREHOUSE_SQL_SCHEMA)
        dataset.autodetect_settings().save()       # the table's columns, read from the database
    else:
        dataset = project.create_dataset(name, "Filesystem", params={"connection": LANDING_CONNECTION,
                                                                     "path": "/" + name},
                                          formatType="csv", formatParams=CSV)
        dataset.autodetect_settings().save()
print(f"{len(simulated)} warehouse tables already in the project, {len(RAW) - len(simulated)} read from {source}")

existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {}
for name in FOLDERS:
    if name in existing_folders:
        folders[name] = project.get_managed_folder(existing_folders[name])
        continue
    folders[name] = project.create_managed_folder(name, connection_name=FOLDER_CONNECTION)
    if name in BUILT_BY_NAME:
        folder_settings = folders[name].get_settings()
        folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
        folder_settings.save()
existing_models = {m["name"]: m["id"] for m in project.list_saved_models()}
saved_model = (project.get_saved_model(existing_models[cashflow.SAVED_MODEL]) if cashflow.SAVED_MODEL in existing_models
               else project.create_mlflow_pyfunc_model(cashflow.SAVED_MODEL, "REGRESSION"))
ref = lambda name: (folders[name].id if name in folders else saved_model.id if name == cashflow.SAVED_MODEL
                    else name)

for name, (inputs, outputs, _) in FLOW.items():
    for item in inputs + outputs:
        if item not in folders and item != cashflow.SAVED_MODEL:
            new_dataset(item)
    if name in recipes:                # already there: its inputs and outputs changed? then made again
        settings = project.get_recipe(name).get_settings()
        if (set(settings.get_flat_input_refs()) != set(map(ref, inputs))
                or set(settings.get_flat_output_refs()) != set(map(ref, outputs))):
            project.get_recipe(name).delete()
            recipes.discard(name)
            print("recipe", name, "has new inputs or outputs: made again")
    if name in recipes:                # already there: only its code and code env are updated
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o in APPENDED)
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

ZONES = {"Raw data": RAW, "Staging": list(warehouse.STAGED), "Training": [], "Inference": [], "Lab": []}
for _, outputs, zone_name in FLOW.values():
    ZONES[zone_name] += [o for o in outputs if o not in ZONES[zone_name]]
# the raw data goes in Dataiku's Default zone, renamed (so there is no empty zone). It exists once another zone does.
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {z: existing_zones.get(z) or flow.create_zone(z) for z in ZONES if z != "Raw data"}
zones["Raw data"] = existing_zones.get("Raw data") or flow.get_default_zone()
if zones["Raw data"].name != "Raw data":
    zone_settings = zones["Raw data"].get_settings()
    zone_settings.name = "Raw data"
    zone_settings.save()


def safe_add(zone, item):
    """Put an item in a zone (already there: nothing to do)."""
    try:
        zone.add_item(item)
    except Exception as e:
        if "already" not in str(e).lower():
            raise


for zone_name, items in ZONES.items():
    for item in items:
        safe_add(zones[zone_name], folders[item] if item in folders else project.get_dataset(item))
safe_add(zones["Inference"], saved_model)    # next to the recipe that uses it
print("Flow ready:", ", ".join(zones))

## 4. First run

Builds the daily flows (the whole history, the first time), trains, tests and deploys a model, runs champion / challenger (with no production model yet,
the first model is promoted, with a warning in promotion_log if its checks fail), then runs the nightly batch:
forecast, archive, log.

In [ ]:
def build(items, mode):
    job = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            job.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            job.with_output(item)
    job.start_and_wait()


started = time.time()
warehouse.set_cutoff()                                          # the first step of both scenarios
build(["daily_flows"], "RECURSIVE_FORCED_BUILD")             # staging: clients and daily flows
build(["experiments"], "NON_RECURSIVE_FORCED_BUILD")         # train, test, track and deploy the first model
build(["promotion_log"], "NON_RECURSIVE_FORCED_BUILD")       # champion / challenger
production = tracking.apply_promotion()                        # activates its version of cash_forecast
build(["inference_log"], "RECURSIVE_FORCED_BUILD")           # forecast, archive, log the run
print(f"First run done in {(time.time() - started) / 60:.1f} minutes: production model {production}")

## 5. Scenarios

- **Nightly batch**: set the cut-off, add the new days to the daily flows, forecast with the production model,
  archive it and log the run. It never trains.
- **Retrain, test and promote**: set the cut-off, add the new days to the daily flows, train, test and deploy a new model with
  the forecaster the project variable `forecaster` names, champion / challenger with the promotion rules, activate
  it if it passes. It does not forecast: the next Nightly batch uses the new active version.

- **Full rebuild of the daily flows** (run by hand, no trigger): empty `daily_flows` and `staging_log`, then read the
  whole history again. For example after a reversal cancels a posting from a day already staged.

Their triggers (daily at 06:00 and monthly on the 5th, Riyadh time) are created switched off; switch them on
in production.

In [ ]:
# Production runs: two independent scenarios. Nightly batch (inference) forecasts with the active version of the saved
# models; Retrain, test and promote (training) adds a version and activates it if it is better, and never forecasts.
# Each starts by setting the cut-off month (a Python step); the staging recipe adds the new days to daily_flows and
# stops if the postings do not reach the cut-off month. A third one, Full rebuild of the daily flows, has no trigger:
# run it by hand to read the whole history again. The tracked models (folder experiments) and the promotion log are only rebuilt when built by name, so the
# nightly batch never retrains by accident.
def step(n, label, what, mode=None):
    common = {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
              "runConditionStatuses": ["SUCCESS", "WARNING"]}
    if isinstance(what, str):           # a Python step: library function to call
        module = "warehouse" if what in ("set_cutoff", "clear_daily_flows") else "tracking"
        return {**common, "type": "custom_python", "params": {
            "script": f"from {module} import {what}\n\n{what}()\n", "envSelection": {"envMode": "INHERIT"},
            "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}
    kind = lambda i: ("MANAGED_FOLDER", folders[i].id) if i in folders else ("DATASET", i)
    builds = [{"type": t, "itemId": item, "partitionsSpec": ""} for t, item in map(kind, what)]
    return {**common, "type": "build_flowitem", "params": {
        "builds": builds, "jobType": mode, "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


NIGHTLY = [
    ("Set the cut-off month", "set_cutoff"),
    ("Forecast with the production model, archive and log it", ["inference_log"], "RECURSIVE_FORCED_BUILD"),
]
SCENARIOS = {
    "Nightly batch": ("After each warehouse load: forecast the next 3 months with the production model (the active "
                      "version of the saved models), archive it and log the run", "daily", NIGHTLY),
    "Retrain, test and promote": (
        "Train and test a new model with the forecaster the project variable forecaster names (tracked in Experiment "
        "Tracking, deployed as a saved model version), and activate it if it beats the production model. It does not "
        "forecast: the Nightly batch uses the active version", "monthly", [
            ("Set the cut-off month", "set_cutoff"),
            ("Prepare the data (daily flows, clients)", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
            ("Train, test and deploy a new model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Promote if better", "apply_promotion"),
        ]),
    "Full rebuild of the daily flows": (
        "Run by hand: empties daily_flows and staging_log, then reads the whole history from the warehouse again "
        "(the nightly runs only read the new days). For example after a reversal cancels a posting of a day already "
        "staged. It does not forecast: run the Nightly batch after it", None, [
            ("Set the cut-off month", "set_cutoff"),
            ("Empty the daily flows", "clear_daily_flows"),
            ("Read the whole history again", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
        ]),
}
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}
for name, (description, schedule, steps) in SCENARIOS.items():
    if name in existing_scenarios:     # run again: update its steps, keep its triggers and run history
        settings = project.get_scenario(existing_scenarios[name]).get_settings()
        settings.get_raw()["description"] = description
        settings.get_raw()["params"]["steps"] = [step(n, *s) for n, s in enumerate(steps)]
        settings.save()
        print(f"Scenario updated: {name}")
        continue
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [step(n, *s) for n, s in enumerate(steps)]}})
    if schedule is None:                           # run by hand only
        print(f"Scenario ready: {name} (no trigger: run it by hand)")
        continue
    settings = scenario.get_settings()
    if schedule == "daily":
        settings.add_daily_trigger(hour=6, timezone="Asia/Riyadh")
    else:
        settings.add_monthly_trigger(day=5, hour=7, timezone="Asia/Riyadh")
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = "no trigger (not in this licence): add it on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. Results

To use another model (for example to roll back), activate its version in the saved model `cash_forecast` and run
the Nightly batch. To change when a model is promoted, change the project variables `promotion_*` (no code change).

In [ ]:
print("Production model (active version of the saved model cash_forecast):", tracking.production_model_id())
print("Promotion log:")
display(dataiku.Dataset("promotion_log").get_dataframe())
print("Test on the last 6 months: TOTAL rows (net per client), category rows, and ALL together, by months ahead:")
display(dataiku.Dataset("test_results").get_dataframe()[
    ["level", "months_ahead", "rows", "wape_model", "wape_baseline", "coverage_p10_p90", "below_p10", "above_p90"]])
final = dataiku.Dataset("final_forecast").get_dataframe()
one = final.loc[final["status"] == "OK", "client_no"].iloc[0]
print(f"Final forecast: {final['client_no'].nunique()} clients, {len(final)} rows. One client, {one}:")
display(final[final["client_no"] == one])
print("Inference log (one row per forecast run):")
display(dataiku.Dataset("inference_log").get_dataframe().sort_values("run_at"))
print("Staging log (one row per staging run: what was read from the warehouse):")
display(dataiku.Dataset("staging_log").get_dataframe().sort_values("run_at"))